# 01 — Data Preparation
Load, clean, and validate CMS Medicare Inpatient and Physician PUF datasets.

**Inputs:** `data/raw/inpatient_hospital_2022.csv`, `data/raw/physician_practitioners_2022.csv`  
**Outputs:** `data/processed/inpatient_clean.csv`, `data/processed/physician_clean.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

---
## 1. Medicare Inpatient Hospital PUF

In [2]:
inp = pd.read_csv(RAW / 'inpatient_hospital_2022.csv', dtype={'Rndrng_Prvdr_Zip5': str}, encoding='latin-1')
print(f'Shape: {inp.shape}')
inp.dtypes

Shape: (145742, 15)


Rndrng_Prvdr_CCN               int64
Rndrng_Prvdr_Org_Name         object
Rndrng_Prvdr_City             object
Rndrng_Prvdr_St               object
Rndrng_Prvdr_State_FIPS        int64
Rndrng_Prvdr_Zip5             object
Rndrng_Prvdr_State_Abrvtn     object
Rndrng_Prvdr_RUCA            float64
Rndrng_Prvdr_RUCA_Desc        object
DRG_Cd                         int64
DRG_Desc                      object
Tot_Dschrgs                    int64
Avg_Submtd_Cvrd_Chrg         float64
Avg_Tot_Pymt_Amt             float64
Avg_Mdcr_Pymt_Amt            float64
dtype: object

In [3]:
inp.head(5)

,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_City,Rndrng_Prvdr_St,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA_Desc,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Tot_Pymt_Amt,Avg_Mdcr_Pymt_Amt
0,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,23,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,25,158541.640000,37331.000000,35332.960000
1,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,24,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,18,107085.333330,25842.666667,23857.944444
2,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,25,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,18,156326.777780,32167.888889,27662.944444
3,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,38,EXTRACRANIAL PROCEDURES WITH CC,19,112085.263160,11568.473684,9993.473684
4,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,39,EXTRACRANIAL PROCEDURES WITHOUT CC/MCC,33,89068.212121,8199.818182,6086.393939


### 1a. Missingness

In [4]:
miss_inp = inp.isnull().sum()
miss_inp[miss_inp > 0]

Rndrng_Prvdr_RUCA         677
Rndrng_Prvdr_RUCA_Desc    677
dtype: int64

### 1b. Duplicates

In [5]:
dup_count = inp.duplicated(subset=['Rndrng_Prvdr_CCN', 'DRG_Cd']).sum()
print(f'Duplicate rows (CCN + DRG): {dup_count:,}')
inp = inp.drop_duplicates(subset=['Rndrng_Prvdr_CCN', 'DRG_Cd'])
print(f'Shape after dedup: {inp.shape}')

Duplicate rows (CCN + DRG): 0
Shape after dedup: (145742, 15)


### 1c. Clean column names & types

In [6]:
inp.columns = [
    'provider_ccn', 'provider_name', 'provider_city', 'provider_state_fips_name',
    'provider_state_fips', 'provider_zip', 'provider_state', 'provider_ruca',
    'provider_ruca_desc', 'drg_code', 'drg_desc', 'total_discharges',
    'avg_covered_charges', 'avg_total_payments', 'avg_medicare_payments'
]

# Ensure numeric payment columns
for col in ['avg_covered_charges', 'avg_total_payments', 'avg_medicare_payments']:
    inp[col] = pd.to_numeric(inp[col], errors='coerce')

# Derived: markup ratio (charges / Medicare payment)
inp['markup_ratio'] = (inp['avg_covered_charges'] / inp['avg_medicare_payments']).round(2)

print(f'Final shape: {inp.shape}')
inp.dtypes

Final shape: (145742, 16)


provider_ccn                  int64
provider_name                object
provider_city                object
provider_state_fips_name     object
provider_state_fips           int64
provider_zip                 object
provider_state               object
provider_ruca               float64
provider_ruca_desc           object
drg_code                      int64
drg_desc                     object
total_discharges              int64
avg_covered_charges         float64
avg_total_payments          float64
avg_medicare_payments       float64
markup_ratio                float64
dtype: object

In [7]:
inp.head(5)

,provider_ccn,provider_name,provider_city,provider_state_fips_name,provider_state_fips,provider_zip,provider_state,provider_ruca,provider_ruca_desc,drg_code,drg_desc,total_discharges,avg_covered_charges,avg_total_payments,avg_medicare_payments,markup_ratio
0,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,23,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,25,158541.640000,37331.000000,35332.960000,4.49
1,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,24,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,18,107085.333330,25842.666667,23857.944444,4.49
2,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,25,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,18,156326.777780,32167.888889,27662.944444,5.65
3,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,38,EXTRACRANIAL PROCEDURES WITH CC,19,112085.263160,11568.473684,9993.473684,11.22
4,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,1.0,Metropolitan area core: primary flow within an...,39,EXTRACRANIAL PROCEDURES WITHOUT CC/MCC,33,89068.212121,8199.818182,6086.393939,14.63


---
## 2. Medicare Physician & Other Practitioners PUF

In [8]:
phy = pd.read_csv(
    RAW / 'physician_practitioners_2022.csv',
    dtype={'Rndrng_Prvdr_Zip5': str, 'HCPCS_Cd': str},
    encoding='latin-1',
    low_memory=False
)
print(f'Shape: {phy.shape}')
phy.dtypes

Shape: (9755427, 28)


Rndrng_NPI                         int64
Rndrng_Prvdr_Last_Org_Name        object
Rndrng_Prvdr_First_Name           object
Rndrng_Prvdr_MI                   object
Rndrng_Prvdr_Crdntls              object
Rndrng_Prvdr_Ent_Cd               object
Rndrng_Prvdr_St1                  object
Rndrng_Prvdr_St2                  object
Rndrng_Prvdr_City                 object
Rndrng_Prvdr_State_Abrvtn         object
Rndrng_Prvdr_State_FIPS           object
Rndrng_Prvdr_Zip5                 object
Rndrng_Prvdr_RUCA                float64
Rndrng_Prvdr_RUCA_Desc            object
Rndrng_Prvdr_Cntry                object
Rndrng_Prvdr_Type                 object
Rndrng_Prvdr_Mdcr_Prtcptg_Ind     object
HCPCS_Cd                          object
HCPCS_Desc                        object
HCPCS_Drug_Ind                    object
Place_Of_Srvc                     object
Tot_Benes                          int64
Tot_Srvcs                        float64
Tot_Bene_Day_Srvcs                 int64
Avg_Sbmtd_Chrg  

In [9]:
phy.head(5)

,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_MI,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_St1,Rndrng_Prvdr_St2,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,...,HCPCS_Desc,HCPCS_Drug_Ind,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt
0,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Hospital observation care on day of discharge,N,F,42,44.0,44,288.934773,76.932045,58.619773,53.307955
1,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Initial hospital observation care per day, typ...",N,F,17,17.0,17,424.804118,144.920000,109.155294,97.278824
2,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Initial hospital observation care per day, typ...",N,F,35,35.0,35,686.564286,189.998857,151.596857,140.733143
3,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Initial hospital inpatient care per day, typic...",N,F,16,16.0,16,894.991250,100.009375,79.264375,78.499375
4,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Initial hospital inpatient care per day, typic...",N,F,12,12.0,12,511.915000,144.201667,112.946667,103.724167


### 2a. Missingness

In [10]:
miss_phy = phy.isnull().sum()
miss_phy[miss_phy > 0]

Rndrng_Prvdr_First_Name     545299
Rndrng_Prvdr_MI            3355198
Rndrng_Prvdr_Crdntls       1049274
Rndrng_Prvdr_St1                 1
Rndrng_Prvdr_St2           7212576
Rndrng_Prvdr_State_FIPS          3
Rndrng_Prvdr_RUCA            11761
Rndrng_Prvdr_RUCA_Desc       11761
dtype: int64

### 2b. Drop non-essential columns with high missingness

In [11]:
# Middle initial, address line 2, credentials — high-miss, not needed for analysis
drop_cols = ['Rndrng_Prvdr_MI', 'Rndrng_Prvdr_St2', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_Crdntls']
phy = phy.drop(columns=drop_cols)
print(f'Shape after column drop: {phy.shape}')

Shape after column drop: (9755427, 24)


### 2c. Duplicates

In [12]:
dup_count = phy.duplicated(subset=['Rndrng_NPI', 'HCPCS_Cd', 'Place_Of_Srvc']).sum()
print(f'Duplicate rows (NPI + HCPCS + Place): {dup_count:,}')
phy = phy.drop_duplicates(subset=['Rndrng_NPI', 'HCPCS_Cd', 'Place_Of_Srvc'])
print(f'Shape after dedup: {phy.shape}')

Duplicate rows (NPI + HCPCS + Place): 0


Shape after dedup: (9755427, 24)


### 2d. Clean column names & types

In [13]:
phy.columns = [
    'npi', 'provider_last_org_name', 'provider_first_name',
    'provider_entity_code', 'provider_city',
    'provider_state', 'provider_state_fips', 'provider_zip',
    'provider_ruca', 'provider_ruca_desc', 'provider_country',
    'provider_type', 'medicare_participating',
    'hcpcs_code', 'hcpcs_desc', 'hcpcs_drug_ind',
    'place_of_service', 'total_beneficiaries', 'total_services',
    'total_bene_day_services', 'avg_submitted_charge',
    'avg_medicare_allowed', 'avg_medicare_payment', 'avg_medicare_standardized'
]

# Ensure numeric payment columns
for col in ['avg_submitted_charge', 'avg_medicare_allowed', 'avg_medicare_payment', 'avg_medicare_standardized']:
    phy[col] = pd.to_numeric(phy[col], errors='coerce')

# Derived: markup ratio (submitted charge / Medicare allowed)
phy['markup_ratio'] = (phy['avg_submitted_charge'] / phy['avg_medicare_allowed']).round(2)

# Filter to US only (drop territories/foreign)
phy = phy[phy['provider_country'] == 'US'].copy()
phy = phy.drop(columns=['provider_country'])

print(f'Final shape: {phy.shape}')
phy.dtypes

Final shape: (9755020, 24)


npi                            int64
provider_last_org_name        object
provider_first_name           object
provider_entity_code          object
provider_city                 object
provider_state                object
provider_state_fips           object
provider_zip                  object
provider_ruca                float64
provider_ruca_desc            object
provider_type                 object
medicare_participating        object
hcpcs_code                    object
hcpcs_desc                    object
hcpcs_drug_ind                object
place_of_service              object
total_beneficiaries            int64
total_services               float64
total_bene_day_services        int64
avg_submitted_charge         float64
avg_medicare_allowed         float64
avg_medicare_payment         float64
avg_medicare_standardized    float64
markup_ratio                 float64
dtype: object

In [14]:
phy.head(5)

,npi,provider_last_org_name,provider_first_name,provider_entity_code,provider_city,provider_state,provider_state_fips,provider_zip,provider_ruca,provider_ruca_desc,...,hcpcs_drug_ind,place_of_service,total_beneficiaries,total_services,total_bene_day_services,avg_submitted_charge,avg_medicare_allowed,avg_medicare_payment,avg_medicare_standardized,markup_ratio
0,1003000126,Enkeshafi,Ardalan,I,Bethesda,MD,24,20817,1.0,Metropolitan area core: primary flow within an...,...,N,F,42,44.0,44,288.934773,76.932045,58.619773,53.307955,3.76
1,1003000126,Enkeshafi,Ardalan,I,Bethesda,MD,24,20817,1.0,Metropolitan area core: primary flow within an...,...,N,F,17,17.0,17,424.804118,144.920000,109.155294,97.278824,2.93
2,1003000126,Enkeshafi,Ardalan,I,Bethesda,MD,24,20817,1.0,Metropolitan area core: primary flow within an...,...,N,F,35,35.0,35,686.564286,189.998857,151.596857,140.733143,3.61
3,1003000126,Enkeshafi,Ardalan,I,Bethesda,MD,24,20817,1.0,Metropolitan area core: primary flow within an...,...,N,F,16,16.0,16,894.991250,100.009375,79.264375,78.499375,8.95
4,1003000126,Enkeshafi,Ardalan,I,Bethesda,MD,24,20817,1.0,Metropolitan area core: primary flow within an...,...,N,F,12,12.0,12,511.915000,144.201667,112.946667,103.724167,3.55


---
## 3. Validation summary

In [15]:
summary = pd.DataFrame({
    'Dataset': ['Inpatient Hospital', 'Physician & Practitioners'],
    'Rows': [len(inp), len(phy)],
    'Columns': [inp.shape[1], phy.shape[1]],
    'Missing (%)': [
        round(inp.isnull().sum().sum() / inp.size * 100, 2),
        round(phy.isnull().sum().sum() / phy.size * 100, 2)
    ],
    'States': [inp['provider_state'].nunique(), phy['provider_state'].nunique()]
})
summary

,Dataset,Rows,Columns,Missing (%),States
0,Inpatient Hospital,145742,16,0.06,51
1,Physician & Practitioners,9755020,24,0.24,61


---
## 4. Save processed files

In [16]:
inp.to_csv(PROCESSED / 'inpatient_clean.csv', index=False)
phy.to_csv(PROCESSED / 'physician_clean.csv', index=False)

for f in PROCESSED.glob('*.csv'):
    print(f'{f.name}: {f.stat().st_size / 1e6:.1f} MB')

physician_clean.csv: 2849.3 MB
inpatient_clean.csv: 38.9 MB
